In [ ]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
TEST_ONLY = True
import random
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim.lr_scheduler import CosineAnnealingLR
import wandb
import pandas as pd
import tqdm

import cv2
import torch


device = "cuda" if torch.cuda.is_available() else "cpu"

# student = InceptionResnetV1(pretrained='vggface2').train().to(device)
# student = torch.nn.Sequential(
#     student,
#     torch.nn.Linear(512, 1024),
#     torch.nn.ReLU(),
#     torch.nn.Linear(1024, 512)
# ).to(device)
# if os.path.exists("student.pth"):
#     student.load_state_dict(torch.load("student.pth", map_location='cpu'))
#     wandb.init(project="student_distill_insight", resume="must", id="lqwdb3s7")
import torch
from onnx2torch import convert
import sys
sys.path.insert(0, "../")
from minusface import MinusBackbone

conversion_model = MinusBackbone(mode='stage1')
conversion_model.load_state_dict(torch.load("../../../../../minusface_stage1.pth", map_location='cpu'))
conversion_model = conversion_model.eval().to(device)

In [ ]:
import torch
model = torch.hub.load('mateuszbuda/brain-segmentation-pytorch', 'unet',
    in_channels=3, out_channels=3, init_features=3, pretrained=False)

In [ ]:
model.load_state_dict(torch.load("model_minus.pth"))

In [ ]:
tf_student = transforms.Compose([
    transforms.Resize((112,112)),
    transforms.Normalize([0.5]*3,[0.5]*3)
])

tf_conv = transforms.Compose([
    transforms.Resize((112,112)),
    transforms.ToTensor()
])

df = pd.read_csv('/path/to/cropped-celeba/Identity_CelebA (2).txt', sep=' ')
df.columns = ["col0","img_name","id"]
 

In [ ]:
import sys
sys.path.append("../partialface")
import processing_utils as util
embedding_root = '/path/to/casia-webface/insight_embeddings'
pre_loaded_teachers = {}
class FaceDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        p = self.paths[idx]
        # embedding_teacher = pre_loaded_teachers[p]  # (1, 512)
        img_s = cv2.imread(p)
        img_s = img_s[..., ::-1]
        img_s = Image.fromarray(img_s.astype("uint8"))
        c_img = tf_conv(img_s)
        return p, c_img
    
import time
import processing_utils as util


def cosine_triplet(a, p, n, margin=0.3):
    a = F.normalize(a, dim=1)
    p = F.normalize(p, dim=1)
    n = F.normalize(n, dim=1)
    d_ap = 1 - (a * p).sum(dim=1)
    d_an = 1 - (a * n).sum(dim=1)
    return torch.clamp(d_ap - d_an + margin, min=0).mean()

def cosine_sim_loss(a, b):
    a = F.normalize(a, dim=1)
    b = F.normalize(b, dim=1)
    return (1 - (a * b).sum(dim=1)).mean()

def loss_fn(s, t):
    s_n = F.normalize(s, dim=1)
    t_n = F.normalize(t, dim=1)
    idx = torch.arange(s.size(0), device=s.device)
    neg = t_n[torch.roll(idx, shifts=1)]
    trip = cosine_triplet(s_n, t_n, neg)
    cos = cosine_sim_loss(s, t) 
    mae = F.l1_loss(s_n, t_n)
    return trip * 10, cos, mae * 10

paths = []
root = '/path/to/casia-webface'
with open("../fracface/index.txt","r") as f:
    lines = f.readlines()
    for line in lines:
        filename, split = line.strip().split()
        if split != "train":
            paths.append(os.path.join(root, filename)) 

In [ ]:
import time

import torch.nn.functional as F

def convert_batch(conv_raw):
    conv_raw = conv_raw.to(device)
    with torch.no_grad():
        out = conversion_model(conv_raw)[5]

    imgs = out.float()

    minv = imgs.amin(dim=(1, 2, 3), keepdim=True)
    maxv = imgs.amax(dim=(1, 2, 3), keepdim=True)
    imgs = (imgs - minv) / (maxv - minv + 1e-6)
    imgs = (imgs - 0.5) / 0.5  # Normalize to [-1, 1]

    return imgs

In [ ]:
model = model.to(device)

In [ ]:

paths = sorted(paths)
random.seed(42)
random.shuffle(paths)
# paths = paths[:1000]
dataset = FaceDataset(paths)  

print("Dataset size:", len(dataset))
val_loader = DataLoader(dataset, batch_size=256, shuffle=False, num_workers=16, pin_memory=True)

os.makedirs("log", exist_ok=True)

gen_images = []
filenames = []
model = model.eval()

with torch.no_grad():
    for fn, conv_raw in tqdm.tqdm(val_loader):
        s_img = convert_batch(conv_raw.to(device))
        s_gen = model(s_img) 
        gen_images.append(s_gen.detach().cpu())
        filenames.append(fn)

In [ ]:
import pylab
from PIL import Image
pylab.imshow(s_gen.detach().cpu()[8].permute(1, 2, 0).numpy())

In [ ]:
        
import pickle
output_name = "minus_web.pkl" 
with open(output_name, "wb") as f:
    pickle.dump({
        "filenames": filenames,
        "gen_images": gen_images,
    }, f)
